In [1]:
#configuration et chargement
!pip install -q pyarabic qalsadi nltk scikit-learn

import glob, os, re, json, random
import numpy as np, pandas as pd, torch
from collections import Counter
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUT_DIR = "/kaggle/working/data"; os.makedirs(OUT_DIR, exist_ok=True)

path = glob.glob("/kaggle/input/**/dataset.csv", recursive=True)
print(path)
df = pd.read_csv(path[0])
print(df.shape, df.columns.tolist())      # attendu : (570, 2) ['text', 'score']

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
['/kaggle/input/datasets/lorraineagns/collecter-un-corpus-de-textes-arabes/dataset.csv']
(570, 2) ['text', 'score']


In [2]:
# nettoyage, tokenisation, stop words
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer

TASHKEEL = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0640]")
URL = re.compile(r"https?://\S+|www\.\S+")

def normalize(t):
    t = TASHKEEL.sub("", t)
    return re.sub("[إأآٱ]", "ا", t).replace("ى", "ي").lower()

def clean(t):
    t = normalize(URL.sub(" ", str(t)))
    t = re.sub(r"[0-9\u0660-\u0669]+(?:[.,][0-9]+)?", " num ", t)     # nombres -> "num"
    return re.sub(r"[^\u0621-\u064Aa-z\s]", " ", t)                   # ponctuation et symboles

STOP = {normalize(w) for w in stopwords.words("arabic")} | {"num"}
def tokenize(t): return [w for w in clean(t).split() if len(w) > 1]

df["tokens"] = df["text"].map(tokenize)
df["tokens_ns"] = df["tokens"].map(lambda ws: [w for w in ws if w not in STOP])
print("tokens par article (médiane), avant / après stop words :",
      int(df["tokens"].map(len).median()), "/", int(df["tokens_ns"].map(len).median()))

tokens par article (médiane), avant / après stop words : 374 / 286


In [3]:
# stemming et lemmatisation
stemmer = ISRIStemmer()
vocab = sorted({w for ws in df["tokens_ns"] for w in ws})
print("mots distincts :", len(vocab))
stem_map = {w: stemmer.stem(w) for w in vocab}

lemma_map = {}
try:
    import qalsadi.lemmatizer
    lemmer = qalsadi.lemmatizer.Lemmatizer()
    for w in tqdm(vocab):
        try: lemma_map[w] = normalize(lemmer.lemmatize(w))
        except Exception: lemma_map[w] = w
except Exception as e:
    print("qalsadi indisponible, repli sur le stemming :", e)
    lemma_map = dict(stem_map)

df["stems"]  = df["tokens_ns"].map(lambda ws: [stem_map[w] for w in ws])
df["lemmas"] = df["tokens_ns"].map(lambda ws: [lemma_map[w] for w in ws])

for col in ["tokens", "tokens_ns", "stems", "lemmas"]:
    print(f"{col:10s} vocabulaire = {len({w for ws in df[col] for w in ws}):6d}")
print(df.loc[3, "tokens_ns"][:10]); print(df.loc[3, "stems"][:10]); print(df.loc[3, "lemmas"][:10])

mots distincts : 40025


/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:274: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN = re.compile(u"([^\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:276: SyntaxWarning: invalid escape sequence '\w'
  TOKEN_PATTERN_SPLIT = re.compile(u"([\w\u0670\u064b-\u0652']+)", re.UNICODE)
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:281: SyntaxWarning: invalid escape sequence '\s'
  ARABIC_STRING = re.compile(u"([^\u0600-\u0652%s%s%s\s\d])" \
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:1237: SyntaxWarning: invalid escape sequence '\s'
  u"(?<=\s(%s|%s))%s" % (WAW, YEH, FATHA), \
/usr/local/lib/python3.13/dist-packages/pyarabic/araby.py:1450: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub(u"(?<=[\s\d])([%s])+"%(TASHKEEL_STRING),"",text,  re.UNICODE)


  0%|          | 0/40025 [00:00<?, ?it/s]

tokens     vocabulaire =  40369
tokens_ns  vocabulaire =  40025
stems      vocabulaire =   8987
lemmas     vocabulaire =  15075
['شهد', 'قطاع', 'الالعاب', 'الالكترونية', 'تطورا', 'واسعا', 'السنوات', 'الماضية', 'بدءا', 'ارتفاع']
['شهد', 'قطع', 'لعب', 'الكترونية', 'تطر', 'وسع', 'سنو', 'اضة', 'بدء', 'رفع']
['شاهد', 'قاطع', 'الالعاب', 'الالكترونية', 'تطور', 'واسع', 'سنة', 'ماضي', 'بدء', 'ارتفاع']


In [4]:
# discrétisation, découpage, encodage, sauvegarde
TOKEN_COL = "lemmas"                 # on pourra tester "stems" dans le notebook 03
df["label"] = pd.cut(df["score"], [-0.1, 2.0, 6.0, 10.0], labels=[0, 1, 2]).astype(int)  # 0 faible, 1 moyen, 2 élevé
print(df["label"].value_counts().sort_index())

idx = np.arange(len(df))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=df["label"], random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=df.loc[tmp, "label"], random_state=SEED)
print("train / val / test :", len(tr), len(va), len(te))

print(df[TOKEN_COL].map(len).describe(percentiles=[.5, .75, .9]).round(0))
MAX_LEN = 256

cnt = Counter(w for i in tr for w in df.loc[i, TOKEN_COL])            # vocabulaire construit sur le train seul
itos = ["<pad>", "<unk>"] + [w for w, c in cnt.most_common() if c >= 2][:20000]
stoi = {w: k for k, w in enumerate(itos)}
print("taille du vocabulaire :", len(itos))

def encode(ws):
    ids = [stoi.get(w, 1) for w in ws][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

X = torch.tensor([encode(ws) for ws in df[TOKEN_COL]], dtype=torch.long)
torch.save({"X": X, "y_cls": torch.tensor(df["label"].values), "y_reg": torch.tensor(df["score"].values, dtype=torch.float32),
            "train": tr, "val": va, "test": te, "itos": itos, "max_len": MAX_LEN, "token_col": TOKEN_COL},
           f"{OUT_DIR}/prep_data.pt")
df[["tokens_ns", "stems", "lemmas", "score", "label"]].to_pickle(f"{OUT_DIR}/dataset_preprocessed.pkl")
print("sauvegardé :", X.shape)

label
0    337
1    118
2    115
Name: count, dtype: int64
train / val / test : 399 85 86
count     570.0
mean      383.0
std       328.0
min        40.0
50%       286.0
75%       489.0
90%       788.0
max      2799.0
Name: lemmas, dtype: float64
taille du vocabulaire : 7762
sauvegardé : torch.Size([570, 256])
